<a href="https://colab.research.google.com/github/Momonikpour/llama3-turkish-finetune-LLM/blob/main/Ince_Ayar_odevi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-cache-dir --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-fl79_xw_/unsloth_01d56d909d9d495395ecb68441e759b1
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-fl79_xw_/unsloth_01d56d909d9d495395ecb68441e759b1
  Resolved https://github.com/unslothai/unsloth.git to commit 7382805debfefb31534e4b3e6b05a553af3aeff3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 208.6 MB/s eta 0:00:00


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto-detects the data type for the hardware
load_in_4bit = True # This shrinks the 8B model so it fits on a Colab T4 GPU

# 1. Load the Base Model and Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit", # Pre-quantized Llama 3 for faster loading
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. Apply LoRA Adapters (The "İnce Ayar" mechanism)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # The rank of the adapter. 16 is a great balance of speed and accuracy.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Setting to 0 is optimized for Unsloth
    bias = "none",    # Setting to "none" is optimized for Unsloth
    use_gradient_checkpointing = "unsloth", # Crucial for saving VRAM during training
    random_state = 3407,
)

print("\nModel and LoRA adapters successfully loaded!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.6: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.
Unsloth 2026.10.6 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.



Model and LoRA adapters successfully loaded!


In [ ]:
from datasets import load_dataset

# 1. Define the Alpaca prompt template
alpaca_prompt = """Aşağıdaki talimatı, ek bir bağlam (giriş) varsa onu da kullanarak yerine getiren bir yanıt yaz.

### Talimat:
{}

### Giriş:
{}

### Yanıt:
{}"""

EOS_TOKEN = tokenizer.eos_token  # Appends end-of-sequence token so the model knows when to stop generating

# 2. Formatting function to map dataset fields into the template
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_text, output) + EOS_TOKEN
        texts.append(text)
    return { "text": texts }

# 3. Load the dataset from Hugging Face
# Using a 3,000 sample slice keeps fine-tuning under ~10-15 minutes on a free T4 GPU
dataset = load_dataset("cenfis/alpaca-turkish-combined", split="train[:3000]")
dataset = dataset.map(formatting_prompts_func, batched=True)

# 4. Quick inspection
print("Sample prompt formatted for training:\n")
print(dataset["text"][0])

README.md:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 25.8MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/82353 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Sample prompt formatted for training:

Aşağıdaki talimatı, ek bir bağlam (giriş) varsa onu da kullanarak yerine getiren bir yanıt yaz.

### Talimat:
Fransa'nın başkenti nedir?

### Giriş:


### Yanıt:
Fransa'nın başkenti Paris'tir.<|end_of_text|>


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Set up the trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training faster for short sequences
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Change to num_train_epochs = 1 for a full real training run
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# Start the fine-tuning!
trainer_stats = trainer.train()

print("\nİnce ayar (Fine-tuning) completed successfully!")

Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/3000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.535896
2,2.453024
3,2.464478
4,2.022190
5,2.124691
6,1.735654
7,1.533335
8,1.544186
9,1.717896
10,1.478858


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.



İnce ayar (Fine-tuning) completed successfully!


In [ ]:
FastLanguageModel.for_inference(model)

# Test için modele daha önce görmediği yeni bir Türkçe talimat verelim
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Yapay zeka modellerinde ince ayar (fine-tuning) ne işe yarar?", # Talimat
        "", # Giriş (boş)
        "", # Yanıt (model üretecek)
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])

Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


<|begin_of_text|>Aşağıdaki talimatı, ek bir bağlam (giriş) varsa onu da kullanarak yerine getiren bir yanıt yaz.

### Talimat:
Yapay zeka modellerinde ince ayar (fine-tuning) ne işe yarar?

### Giriş:


### Yanıt:
Yapay zeka modellerini ince ayarlamak, modeli bir belge koleksiyonuna özel hale getirmeye ve daha iyi sonuçlar elde etmeye yardımcı olabilir. Bu, daha iyi performans, daha yüksek doğruluk ve daha iyi genelizasyon için bir yol olabilir.<|end_of_text|>


In [ ]:
FastLanguageModel.for_inference(model)

print("--- Türkçe Alpaca Modeli Canlı Test Arayüzü ---")
print("Çıkmak için talimata 'çıkış' veya 'exit' yazın.\n")

while True:
    instruction = input("\n[Talimat girin]: ").strip()
    if instruction.lower() in ["çıkış", "exit", "quit", "q"]:
        print("Arayüz sonlandırıldı.")
        break
    if not instruction:
        continue

    input_text = input("[Giriş/Bağlam girin (boş geçebilirsiniz)]: ").strip()

    # Alpaca şablonu oluşturma
    formatted_prompt = alpaca_prompt.format(
        instruction,
        input_text,
        "" # Modelin doldurması için yanıt alanı boş bırakılır
    )

    inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )

    # Çıktıdan sadece üretilen yanıt kısmını ayıklama
    decoded = tokenizer.batch_decode(outputs)[0]
    response = decoded.split("### Yanıt:\n")[-1].replace(tokenizer.eos_token, "").strip()

    print("\n--- Modelin Yanıtı ---")
    print(response)
    print("----------------------")

--- Türkçe Alpaca Modeli Canlı Test Arayüzü ---
Çıkmak için talimata 'çıkış' veya 'exit' yazın.


[Talimat girin]: Evde ders çalışırken odaklanmayı artırmak için 3 pratik öneri ver
[Giriş/Bağlam girin (boş geçebilirsiniz)]: 


Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Modelin Yanıtı ---
1. Çalışma alanını düzenleyin: Çalışma alanınızı düzenleyerek, odaklanmayı artırmak için iyi bir pratik olabilir. Çalışma alanını düzenlemek için, bilgisayarınızın ve diğer gereçlerin etrafında bir düzen oluşturun. Ayrıca, çalışma alanınızı düzenlemek için bir çalışma masası, bir çalışma odası veya bir çalışma alanı oluşturabilirsiniz.

2. Çalışma alanınızdaki distraktif unsurları azaltın: Çalışma alanınızdaki distraktif unsurları azaltarak, odaklanmayı artırmak için iyi bir
----------------------

[Talimat girin]: iranin baskenti nedir 
[Giriş/Bağlam girin (boş geçebilirsiniz)]: 


Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Modelin Yanıtı ---
Iran'ın başkenti Tahran'dır.
----------------------

[Talimat girin]: bir kopekde 2 top var 
[Giriş/Bağlam girin (boş geçebilirsiniz)]: 


...

Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Modelin Yanıtı ---
Harika.
----------------------

[Talimat girin]: 

[Talimat girin]: 	 Fransa'nın başkenti nedir?
[Giriş/Bağlam girin (boş geçebilirsiniz)]: 	 Fransa'nın başkenti nedir?


Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Modelin Yanıtı ---
Fransa'nın başkenti Paris'tir.
----------------------


KeyboardInterrupt: Interrupted by user